In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader,Dataset,ConcatDataset
import numpy as np
import pandas as pd

In [141]:
df_test=pd.read_csv("../dataset/test.csv")

In [142]:
df_test.head()

,time,channel,PRN,Carrier_Doppler_hz,Pseudorange_m,RX_time,TOW,Carrier_phase,EC,LC,PC,PIP,PQP,TCD,CN0
0,111402,ch0,8,4749.068417,4.841489e+06,263154.82,263154.803851,-435396.111594,101998.429688,100788.812500,111415.289062,-107731.039062,-28414.615234,4660.467773,43.559540
1,111402,ch1,9,1995.777378,2.449848e+06,263154.82,263154.811828,-201479.950180,100812.039062,98424.367188,109174.476562,-105347.234375,-28653.570312,1997.164185,47.116425
2,111402,ch2,27,3458.024120,3.738822e+06,263154.82,263154.807529,-356000.635312,138335.140625,125640.570312,138446.281250,137254.781250,-18124.476562,3467.803955,45.687332
3,111402,ch3,26,-2105.992470,5.482371e+06,263154.82,263154.801713,193102.504674,84470.195312,84153.156250,95917.468750,95899.140625,1874.903442,-2109.972656,42.264256
4,111402,ch4,4,-493.638193,2.666526e+06,263154.82,263154.811105,48203.346004,103226.210938,100365.203125,114574.656250,-114160.539062,-9732.571289,-494.215881,43.629795


In [143]:
df_test.info()

<class 'pandas.DataFrame'>
RangeIndex: 381952 entries, 0 to 381951
Data columns (total 15 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   time                381952 non-null  int64  
 1   channel             381952 non-null  str    
 2   PRN                 381952 non-null  int64  
 3   Carrier_Doppler_hz  381952 non-null  float64
 4   Pseudorange_m       381952 non-null  float64
 5   RX_time             381952 non-null  float64
 6   TOW                 381952 non-null  float64
 7   Carrier_phase       381952 non-null  float64
 8   EC                  381952 non-null  float64
 9   LC                  381952 non-null  float64
 10  PC                  381952 non-null  float64
 11  PIP                 381952 non-null  float64
 12  PQP                 381952 non-null  float64
 13  TCD                 381952 non-null  float64
 14  CN0                 381952 non-null  float64
dtypes: float64(12), int64(2), str(1)
memory usage

In [144]:
df_train = pd.read_csv("../dataset/train.csv")
df_train

/tmp/ipykernel_33440/1898863266.py:1: DtypeWarning: Columns (0: PRN, 1: Carrier_Doppler_hz, 2: Pseudorange_m, 3: RX_time, 4: TOW, 5: Carrier_phase, 6: EC, 7: LC, 8: PC, 9: PIP, 10: PQP, 11: TCD, 12: CN0) have mixed types. Specify dtype option on import or set low_memory=False.
  df_train = pd.read_csv("../dataset/train.csv")


,time,channel,PRN,Carrier_Doppler_hz,Pseudorange_m,RX_time,TOW,Carrier_phase,EC,LC,PC,PIP,PQP,TCD,CN0,spoofed
0,0,ch0,ch0,ch0,ch0,ch0,ch0,ch0,ch0,ch0,ch0,ch0,ch0,ch0,ch0,0
1,0,ch1,ch1,ch1,ch1,ch1,ch1,ch1,ch1,ch1,ch1,ch1,ch1,ch1,ch1,0
2,0,ch2,ch2,ch2,ch2,ch2,ch2,ch2,ch2,ch2,ch2,ch2,ch2,ch2,ch2,0
3,0,ch3,ch3,ch3,ch3,ch3,ch3,ch3,ch3,ch3,ch3,ch3,ch3,ch3,ch3,0
4,0,ch4,ch4,ch4,ch4,ch4,ch4,ch4,ch4,ch4,ch4,ch4,ch4,ch4,ch4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
891211,111401,ch3,26,-2113.491414,5482363.057687,263154.8,263154.781713,193060.42721,101265.445312,96612.585938,104833.25,-104328.625,-10273.680664,-2108.602539,42.208794,0
891212,111401,ch4,4,-493.836685,2666524.741959,263154.8,263154.791105,48193.433748,112036.953125,122600.25,124538.890625,124537.945312,-485.292389,-495.879303,43.671265,0
891213,111401,ch5,0,0.0,0.0,263154.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
891214,111401,ch6,0,0.0,0.0,263154.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [145]:
cols=[]
for col in df_train.columns:
    if df_train[col].dtype == "object":
        print(col)
        cols.append(col)
df_train[cols] = df_train[cols].apply(pd.to_numeric, errors='coerce')

PRN
Carrier_Doppler_hz
Pseudorange_m
RX_time
TOW
Carrier_phase
EC
LC
PC
PIP
PQP
TCD
CN0


In [146]:
df_train

,time,channel,PRN,Carrier_Doppler_hz,Pseudorange_m,RX_time,TOW,Carrier_phase,EC,LC,PC,PIP,PQP,TCD,CN0,spoofed
0,0,ch0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,0,ch1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,0,ch2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
3,0,ch3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
4,0,ch4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
891211,111401,ch3,26.0,-2113.491414,5.482363e+06,263154.8,263154.781713,193060.427210,101265.445312,96612.585938,104833.250000,-104328.625000,-10273.680664,-2108.602539,42.208794,0
891212,111401,ch4,4.0,-493.836685,2.666525e+06,263154.8,263154.791105,48193.433748,112036.953125,122600.250000,124538.890625,124537.945312,-485.292389,-495.879303,43.671265,0
891213,111401,ch5,0.0,0.000000,0.000000e+00,263154.8,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0
891214,111401,ch6,0.0,0.000000,0.000000e+00,263154.8,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0


In [147]:
df_train = df_train.tail(100*8)
df_train

,time,channel,PRN,Carrier_Doppler_hz,Pseudorange_m,RX_time,TOW,Carrier_phase,EC,LC,PC,PIP,PQP,TCD,CN0,spoofed
890416,111302,ch0,8.0,4749.919566,4.843298e+06,263152.82,263152.803844,-425894.714904,105991.023438,115009.265625,117640.156250,-115539.093750,22134.250000,4650.250488,43.408607,0
890417,111302,ch1,9.0,1995.847806,2.450607e+06,263152.82,263152.811826,-197487.623925,106948.375000,109427.648438,123364.406250,-111236.726562,-53340.101562,2002.302124,46.963428,0
890418,111302,ch2,27.0,3459.985886,3.740137e+06,263152.82,263152.807524,-349082.510797,124562.070312,120765.781250,132095.656250,131227.031250,15123.750000,3468.656250,46.216827,0
890419,111302,ch3,26.0,-2099.807104,5.481577e+06,263152.82,263152.801715,188895.902829,64256.984375,102934.445312,89087.953125,-83185.601562,-31887.593750,-2105.559814,41.496189,0
890420,111302,ch4,4.0,-498.066040,2.666341e+06,263152.82,263152.811106,47210.670425,67283.601562,71301.828125,80015.703125,79501.687500,9055.051758,-493.369720,44.639210,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
891211,111401,ch3,26.0,-2113.491414,5.482363e+06,263154.80,263154.781713,193060.427210,101265.445312,96612.585938,104833.250000,-104328.625000,-10273.680664,-2108.602539,42.208794,0
891212,111401,ch4,4.0,-493.836685,2.666525e+06,263154.80,263154.791105,48193.433748,112036.953125,122600.250000,124538.890625,124537.945312,-485.292389,-495.879303,43.671265,0
891213,111401,ch5,0.0,0.000000,0.000000e+00,263154.80,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0
891214,111401,ch6,0.0,0.000000,0.000000e+00,263154.80,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0


In [148]:
df = pd.concat([df_train,df_test])

In [149]:
mapping = {f"ch{i}":i for i in range(0,8)}
df["channel"] = df["channel"].map(mapping)

In [150]:
nn_dfs=[]

for i in range(0,8):
    temp = df[df["channel"]==i].copy()
    temp = temp.sort_values("time")
    nn_dfs.append(temp)

In [151]:
import joblib
single_row_model = joblib.load("xgb_single_row_model.pkl")

In [152]:
df_test["channel"] = df_test["channel"].map(mapping)

In [153]:
xgb_dfs=[]

for i in range(0,8):
    temp = df_test[df_test["channel"]==i].copy()
    temp = temp.sort_values("time")
    xgb_dfs.append(temp)

In [154]:
times = df_test.time.unique()

In [155]:
i = 0
result = None

for dataframe in xgb_dfs:
    # Drop the 'time' column if it exists
    dataframe = dataframe.drop(columns=["time"])  # or dataframe.drop(["time"], axis=1)
    
    # Get predictions for all rows
    y_proba = single_row_model.predict_proba(dataframe)[:, 1]  # Get probability for class 1
    
    # Create DataFrame with predictions
    temp_df = pd.DataFrame({
        "time":times,
        "confidence": y_proba,
        "channel": i
    })
    
    i += 1
    
    if result is None:
        result = temp_df
    else:
        result = pd.concat([result, temp_df], ignore_index=True)

In [156]:
result = result.sort_values("time")

In [196]:
class CrudeDataset(Dataset):
    def __init__(self, df, seq_len=100):
        self.df = df.copy()  # Create a copy to avoid warnings
        self.seq_len = seq_len
        
        # Ensure correct types
        self.df["channel"] = self.df["channel"].astype(int)
        
        # Sort by time
        self.df = self.df.sort_values("time")
        
        # Create valid indices where we have enough sequence length
        self.valid_indices = []
        for idx in range(len(self.df) - seq_len ):  # Need seq_len + 1 for features + target
            self.valid_indices.append(idx)
        
    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        # Get actual index from valid_indices
        actual_idx = self.valid_indices[idx]
        
        # Get sequence of seq_len + 1 rows (for features + target)
        batch_data = self.df.iloc[actual_idx:actual_idx + self.seq_len + 1]
        
        # Check if we have enough data
        if len(batch_data) < self.seq_len + 1:
            # Pad if needed (should not happen with valid_indices)
            pad_needed = (self.seq_len + 1) - len(batch_data)
            last_row = batch_data.iloc[-1:].copy()
            for _ in range(pad_needed):
                batch_data = pd.concat([batch_data, last_row], ignore_index=True)
        
        # Get target info (last row)
        target_row = batch_data.iloc[-1]
        target_time = target_row["time"]
        channel = target_row["channel"]
        
        # Define target variable (you need to specify what to predict)
        # Assuming you have a 'target' column or want to predict something        
        # Drop unused columns
        batch_data = batch_data.drop(columns=["time", "channel"])  # Add other columns to drop
        
        # Convert to numpy
        data = batch_data.to_numpy(dtype=np.float32)
        
        # Split into features and transformed_target
        features = data[:-1]  # (seq_len, F)
        transformed_target = data[-1]  # (F,)
        
        # Ensure fixed sequence length
        if len(features) < self.seq_len:
            pad = np.zeros((self.seq_len - len(features), features.shape[1]), dtype=np.float32)
            features = np.vstack([pad, features])
        else:
            features = features[-self.seq_len:]
        
        # Convert to tensors
        features = torch.from_numpy(features)
        transformed_target = torch.from_numpy(transformed_target)
        
        # Numerical safety
        features = torch.nan_to_num(features, nan=0.0, posinf=10.0, neginf=-10.0)
        transformed_target = torch.nan_to_num(transformed_target, nan=0.0, posinf=10.0, neginf=-10.0)
        
        # Clamp
        features = torch.clamp(features, -5.0, 5.0)
        transformed_target = torch.clamp(transformed_target, -5.0, 5.0)
        
        return features, transformed_target, target_time, channel

In [197]:
combined_dataset = []

for df in nn_dfs:
    print(df.shape)
    dataset=CrudeDataset(df)
    combined_dataset.append(dataset)

(47844, 16)
(47844, 16)
(47844, 16)
(47844, 16)
(47844, 16)
(47844, 16)
(47844, 16)
(47844, 16)


In [198]:
combined_dataset = ConcatDataset(combined_dataset)


In [199]:
loader = DataLoader(combined_dataset, batch_size=32, shuffle=False)

In [200]:
features, transformed_target,time,channel = next(iter(loader))
print("Time shape:", time.shape)  # (batch_size, 1)
print("Features shape:", features.shape)  # (batch_size, seq_len, num_features)
print("Transformed Target shape:", transformed_target.shape)  # (batch_size, num_features)
print("Channel shape:", channel.shape)

Time shape: torch.Size([32])
Features shape: torch.Size([32, 100, 14])
Transformed Target shape: torch.Size([32, 14])
Channel shape: torch.Size([32])


In [168]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, dropout=0.1, max_len=5000):
        super(PositionalEncoding, self).__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        
        self.register_buffer('pe', pe)
        
    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class TimeSeriesTransformer(nn.Module):
    def __init__(self, input_dim=14, d_model=128, nhead=8, num_layers=3, 
                 dim_feedforward=256, dropout=0.1,):
        super(TimeSeriesTransformer, self).__init__()
        
        self.d_model = d_model
                
        # Input projection
        self.input_projection = nn.Linear(input_dim, d_model)
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout)
        
        # Transformer encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
                
        # Output layers
        self.layer_norm1 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)
        
        # Prediction head
        self.output_projection = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, input_dim)
        )
        
    def forward(self, x):
        batch_size, seq_len, input_dim = x.shape
        # Reshape for transformer
        outputs= {
            'input': x,
            'projected': None,
            'positional_encoded': None,
            'encoder_outputs': [],
            "before_output_projection": None,
            'final_output': None
        }
        x = x.view(batch_size , seq_len, input_dim)
        
        # Project input
        x = self.input_projection(x) * math.sqrt(self.d_model)
        outputs['projected'] = x.detach().cpu()
        # Add positional encoding
        x = self.pos_encoder(x)
        outputs['positional_encoded'] = x.detach().cpu()
        
        residual = x[:, -1, :]  # (batch, input_dim)

        # Apply transformer
        x = self.transformer(x)  # (batch, seq_len, d_model)
        outputs['encoder_outputs'] = x.detach().cpu()
        
        # Take last timestep
        x = x[:, -1, :]  # (batch, d_model)
        
        # Reshape back
        x = x.view(batch_size, self.d_model)
        
        # Apply layer norm and residual
        x = self.layer_norm1(x+residual)
        x = self.dropout(x)

        outputs["before_output_projection"] = x.detach().cpu()
        
        # Output projection
        output = self.output_projection(x)
        outputs['final_output'] = output.detach().cpu()
        
        return output, outputs

class CNNBlock(nn.Module):
    """CNN block for binary classification after transformer output"""
    def __init__(self, in_channels, hidden_channels=256, dropout=0.1):
        super(CNNBlock, self).__init__()
        
        self.conv_layers = nn.Sequential(
            # First conv block
            nn.Conv2d(in_channels, hidden_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_channels),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            
            # Second conv block
            nn.Conv2d(hidden_channels, hidden_channels // 2, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_channels // 2),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
            
            # Third conv block
            nn.Conv2d(hidden_channels // 2, hidden_channels // 4, kernel_size=3, padding=1),
            nn.BatchNorm2d(hidden_channels // 4),
            nn.ReLU(inplace=True),
            nn.Dropout2d(dropout),
        )
        
        # Global average pooling and classification head
        self.global_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.classifier = nn.Sequential(
            nn.Linear(hidden_channels // 4, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, 1)  # Binary classification output
        )
        
    def forward(self, x):
        # x shape: (batch, channels=nhead*num_layers, seq_len, d_model)
        x = self.conv_layers(x)
        x = self.global_pool(x)
        x = x.view(x.size(0), -1)  # Flatten
        cnn_features = x  # Save features for potential use
        x = self.classifier(x)
        return x, cnn_features


class ImageAnalysis(nn.Module):

    def __init__(self, input_dim=14, d_model=128, nhead=8, num_layers=3, 
                 dim_feedforward=256, dropout=0.1, cnn_hidden=256):
        super(ImageAnalysis, self).__init__()

        self.d_model = d_model
        self.nhead = nhead
        self.num_layers = num_layers

        # Input projection
        self.input_projection = nn.Linear(input_dim, d_model)

        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, dropout)

        # Transformer encoder
        self.encoder_layers = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=d_model,
                nhead=nhead,
                dim_feedforward=dim_feedforward,
                dropout=dropout,
                batch_first=True
            )
            for _ in range(num_layers)
        ])

        self.to_image = nn.Linear(d_model, nhead * d_model)
        
        # CNN block for binary classification
        # Input channels = nhead * num_layers (concatenated across transformer layers)
        self.cnn_block = CNNBlock(
            in_channels=nhead * num_layers,  # FIXED: multiplied by num_layers
            hidden_channels=cnn_hidden,
            dropout=dropout
        )

    def forward(self, x):
        batch_size, seq_len, input_dim = x.shape

        images=[]

        x = x.view(batch_size, seq_len, input_dim)

        # Project input
        x = self.input_projection(x) * math.sqrt(self.d_model)

        # Add positional encoding
        x = self.pos_encoder(x)

        # Apply transformer
        for layer_idx, encoder_layer in enumerate(self.encoder_layers):
            x = encoder_layer(x)

            image = self.to_image(x)  # (batch, seq_len, nhead*d_model)
            image = image.view(batch_size, seq_len, self.nhead, self.d_model)
            image = image.permute(0, 2, 1, 3)  # (batch, nhead, seq_len, d_model)
            images.append(image)

        # Concatenate all transformer layer outputs
        # Shape: (batch, nhead * num_layers, seq_len, d_model)
        result = torch.cat(images, dim=1)
        
        # Apply CNN block for binary classification
        cnn_output, cnn_features = self.cnn_block(result)
        
        final_output = torch.sigmoid(cnn_output)
        
        return final_output, cnn_features


In [203]:
# Initialize models
timeseries_model = TimeSeriesTransformer()
image_analysis_model = ImageAnalysis(
    input_dim=14,
    d_model=64,
    nhead=4,
    num_layers=2,
    dim_feedforward=128,
    dropout=0.3, 
    cnn_hidden=128
)

In [204]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [205]:

checkpoint = torch.load('finetuned_with_head.pth', map_location=device)
timeseries_model.load_state_dict(checkpoint['timeseries_model'])
image_checkpoint = torch.load('best_cnn_model.pth', map_location=device)
image_analysis_model.load_state_dict(image_checkpoint)

<All keys matched successfully>

In [206]:
timeseries_model = timeseries_model.to(device)
image_analysis_model = image_analysis_model.to(device)

In [207]:
timeseries_model.eval()

X_transf_block = []
time_transf_block = []
X_image_block = []
channels=[]

i=0
with torch.no_grad():
    for features, transformed_target,time,channel in loader:
        i+=1
        features = features.to(device)
        transformed_target = transformed_target.to(device)

        # Extract representation
        ts_output, outputs = timeseries_model(features)   # (B, 15)

        image_input = torch.cat([features, transformed_target.unsqueeze(1)], dim=1)  # (batch, seq_len, num_features + target_dim)
        image_input = image_input.to(device)
        y_pred,features = image_analysis_model(image_input)

        temp = outputs["before_output_projection"]
        temp = torch.tensor(temp, dtype=torch.float32).to(device)

        # Choose fusion strategy
        fusion = torch.cat([ts_output, transformed_target.to(device), temp], dim=-1)  # (B, 28)
        # Move to CPU numpy
        X_transf_block.append(fusion.cpu().numpy())
        time_transf_block.append(time.numpy())

        temp = torch.cat([transformed_target, features], dim=1)
        X_image_block.append(temp.detach().cpu().numpy())
        channels.append(channel)
        if i%100==0:
            print(i)

X_tranf = np.concatenate(X_transf_block, axis=0)
X_image = np.concatenate(X_image_block, axis=0)
time = np.concatenate(time_transf_block, axis=0).reshape(-1)
channels = np.concatenate(channels, axis=0).reshape(-1)

print("Transfoemer Feature shape:", X_tranf.shape)  # (N, 15)
print("CNN Features shape:", X_image.shape)
print("time shape:", time.shape)
print("channel shape:", channels)

/tmp/ipykernel_33440/807911823.py:23: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  temp = torch.tensor(temp, dtype=torch.float32).to(device)


100
200
300
400
500
600
700
800
900
1000
1100
1200
1300
1400
1500
1600
1700
1800
1900
2000
2100
2200
2300
2400
2500
2600
2700
2800
2900
3000
3100
3200
3300
3400
3500
3600
3700
3800
3900
4000
4100
4200
4300
4400
4500
4600
4700
4800
4900
5000
5100
5200
5300
5400
5500
5600
5700
5800
5900
6000
6100
6200
6300
6400
6500
6600
6700
6800
6900
7000
7100
7200
7300
7400
7500
7600
7700
7800
7900
8000
8100
8200
8300
8400
8500
8600
8700
8800
8900
9000
9100
9200
9300
9400
9500
9600
9700
9800
9900
10000
10100
10200
10300
10400
10500
10600
10700
10800
10900
11000
11100
11200
11300
11400
11500
11600
11700
11800
11900
Transfoemer Feature shape: (381952, 156)
CNN Features shape: (381952, 46)
time shape: (381952,)
channel shape: [0. 0. 0. ... 7. 7. 7.]


In [208]:
time.shape

(381952,)

In [210]:
time[-1]

np.float64(159145.0)

In [211]:
xgb_tranf=joblib.load("transformer_model_xgb.pkl")
image_analysis_model=joblib.load("image_model_xgb.pkl")

In [212]:
y_proba_tranf=xgb_tranf.predict_proba(X_tranf)
y_proba_image=image_analysis_model.predict_proba(X_image)

In [213]:
new_rs=pd.DataFrame(
    {
        "time":time,
        "channel":channels,
        "tranf_confidence":y_proba_tranf[:,1],
        "image_confidence":y_proba_image[:,1]
    }
)

In [214]:
new_rs=new_rs.sort_values("time")
new_rs.head()

,time,channel,tranf_confidence,image_confidence
334208,111402.0,7.0,0.047291,0.006530
286464,111402.0,6.0,0.047291,0.006775
190976,111402.0,4.0,0.344388,0.002551
238720,111402.0,5.0,0.047291,0.003768
47744,111402.0,1.0,0.069774,0.169167


In [215]:
result

,time,confidence,channel
334208,111402,0.000475,7
286464,111402,0.000484,6
190976,111402,0.000007,4
238720,111402,0.000489,5
47744,111402,0.000010,1
...,...,...,...
334207,159145,0.000010,6
143231,159145,0.000011,2
190975,159145,0.000561,3
95487,159145,0.000010,1


In [216]:
new_rs.shape

(381952, 4)

In [217]:
result.shape

(381952, 3)

In [218]:
merged = pd.merge(result, new_rs, how="outer")

In [220]:
merged.columns

Index(['time', 'confidence', 'channel', 'tranf_confidence',
       'image_confidence'],
      dtype='str')

In [221]:
merged['final_confidence'] = (0.5 * merged['confidence']) + (0.25 * merged['tranf_confidence']) + (0.25 * merged['image_confidence'])

In [222]:
merged['spoofed'] = np.where(merged['final_confidence'] > 0.5, 1, 0)

In [227]:
merged=merged[["time","channel","final_confidence","spoofed"]]
merged

,time,channel,final_confidence,spoofed
0,111402,0,0.135162,0
1,111402,1,0.059740,0
2,111402,2,0.092534,0
3,111402,3,0.227687,0
4,111402,4,0.086738,0
...,...,...,...,...
381947,159145,3,0.013540,0
381948,159145,4,0.012507,0
381949,159145,5,0.096343,0
381950,159145,6,0.128934,0


In [228]:
merged = merged.rename(columns={'final_confidence': 'confidence'})
merged

,time,channel,confidence,spoofed
0,111402,0,0.135162,0
1,111402,1,0.059740,0
2,111402,2,0.092534,0
3,111402,3,0.227687,0
4,111402,4,0.086738,0
...,...,...,...,...
381947,159145,3,0.013540,0
381948,159145,4,0.012507,0
381949,159145,5,0.096343,0
381950,159145,6,0.128934,0


In [229]:
merged.to_csv("../dataset/result.csv",index=False)